In [44]:
    # h_daily, c_daily = torch.sum(h_daily, axis=0), torch.sum(c_daily, axis=0)
        
    #     att_weights = torch.unsqueeze(torch.nn.functional.softmax(self.att_daily(x_encoder).squeeze(), dim=1), dim=-1)
    #     h_daily_att = att_weights*x_encoder
    #     h_daily_att = torch.sum(h_daily_att, dim=1)

In [15]:
import math
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch import optim
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from torch.utils.data import Subset
import wandb
wandb.login()

True

In [17]:
!wandb login

wandb: Currently logged in as: dwij (dwij-university-of-minnesota) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [2]:
# Reproducibility
RND_SEED = 42
np.random.seed(RND_SEED)
torch.manual_seed(RND_SEED)

In [3]:
INPUT_NPY = "/users/6/mehta423/daycent/data/processed/daycent_inputs.npy"
OUTPUT_NPY = "/users/6/mehta423/daycent/data/processed/daycent_outputs.npy"
INIT_COND = "/users/6/mehta423/daycent/data/SAS_KGML_090925/InputData/initial_site_conditions.xlsx"

In [4]:
def month_day_ranges():
    mdays = [31,28,31,30,31,30,31,31,30,31,30,31]
    starts, ends = [], []
    s = 0
    for m in mdays:
        starts.append(s)
        ends.append(s + m)   # exclusive
        s += m
    return list(zip(starts, ends))   # 0-indexed day ranges

In [11]:
class DayCentDataset(Dataset):
    def __init__(self, input_npy_path, output_npy_path, init_cond_path, apply_scaling=True, year_emb_dim=16):
        data_dict = np.load(input_npy_path, allow_pickle=True).item()
        self.data = data_dict["data"]        # (N, 365, #features)
        self.mapping = data_dict["mapping"]  # (N, 2) => (point_id, year)
        self.columns = list(data_dict["columns"])

        self.init_conditions = self._load_initial_conditions(init_cond_path)

        # Fit scaler only on Tmin/Tmax/Precip
        temp_idx = [i for i, c in enumerate(self.columns) if c in ["Tmin", "Tmax", "Precip"]]
        if apply_scaling:
            self.scaler = StandardScaler()
            self.scaler.fit(self.data[:, :, temp_idx].reshape(-1, len(temp_idx)))
        else:
            self.scaler = None

        self.year_emb_dim = year_emb_dim

        data_dict = np.load(output_npy_path, allow_pickle=True).item()
        self.somsc = data_dict["somsc"]
        self.cgrain = data_dict["cgrain"]

    def _load_initial_conditions(self, path):
        import pandas as pd
        df = pd.read_excel(path).set_index("id")
        df.dropna(axis=1, inplace=True)  # drop columns that are all NaN
        nan_counts = df.isna().sum()
        print(len(nan_counts[nan_counts > 0]))
        #normalise all columns except 'id'
        cols_to_norm = [c for c in df.columns if c != 'id']
        scaler = StandardScaler()
        df[cols_to_norm] = scaler.fit_transform(df[cols_to_norm])

        return df

    def _year_pos_enc(self, year):
        """Sin-cos positional encoding for year."""
        year_rel = year.astype(int) - 2000
        d = self.year_emb_dim
        pe = np.zeros(d)
        for i in range(0, d, 2):
            div = np.power(10000, 2 * i / d)
            pe[i] = np.sin(year_rel / div)
            pe[i+1] = np.cos(year_rel / div)
        return pe

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq = self.data[idx].copy()
        pid, year = self.mapping[idx]

         # ---- harvest mask ----
        harvest_idx = np.where(seq[:, self.columns.index("harvest_grain")] == 1)[0]
        if len(harvest_idx) > 0:
            cutoff = harvest_idx[0]  # first harvest day
        else:
            cutoff = 364             # if no harvest, allow whole year
        harvest_mask = np.zeros(365, dtype=np.float32)
        harvest_mask[:cutoff+1] = 1.0

        # --- process doy ---
        doy_idx = self.columns.index("doy")
        doy = seq[:, doy_idx]
        doy_sin = np.sin(2 * np.pi * doy / 365)
        doy_cos = np.cos(2 * np.pi * doy / 365)

        # --- process temps/precip ---
        t_idx = [self.columns.index(c) for c in ["Tmin", "Tmax", "Precip"]]
        if self.scaler is not None:
            seq[:, t_idx] = self.scaler.transform(seq[:, t_idx])

        # --- drop old columns and replace with encoded ---
        keep_idx = [i for i, c in enumerate(self.columns) if c not in ["doy", "Tmin", "Tmax", "Precip"]]
        seq = np.concatenate([
            seq[:, keep_idx], 
            doy_sin[:, None], doy_cos[:, None], 
            seq[:, t_idx]  # standardized Tmin/Tmax/Precip
        ], axis=1)

        # --- initial site conditions ---
        init_cond = self.init_conditions.loc[pid.astype(int)].to_numpy().astype(np.float32)

        # --- year encoding ---
        year_pe = self._year_pos_enc(year).astype(np.float32)

        # ---- labels ----
        somsc = self.somsc[idx]            # shape (12,)
        somsc_mask = ~np.isnan(somsc)      # True = valid
        somsc = np.nan_to_num(somsc, nan=0.0)   # replace NaNs with 0 (ignored by mask)

        yield_val = self.cgrain[idx]   # scalar
        yield_mask = ~np.isnan(yield_val)  # bool
        if np.isnan(yield_val):
            yield_val = 0.0

        return {
        "sequence": torch.tensor(seq, dtype=torch.float32),
        "init_cond": torch.tensor(init_cond, dtype=torch.float32),
        "year_enc": torch.tensor(year_pe, dtype=torch.float32),
        "somsc": torch.tensor(somsc, dtype=torch.float32),
        "somsc_mask": torch.tensor(somsc_mask.astype(np.float32)),
        "yield": torch.tensor(yield_val, dtype=torch.float32),
        "yield_mask": torch.tensor(float(yield_mask)),
        "harvest_mask": torch.tensor(harvest_mask, dtype=torch.float32),
        "pid": pid,
        "year": year
    }

In [12]:
# check dataset
dataset = DayCentDataset(INPUT_NPY, OUTPUT_NPY, INIT_COND, apply_scaling=True)

0


In [18]:
class AttentionPooling(nn.Module):
    """Generic attention pooling over time dimension."""
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.attn = nn.Linear(input_dim, 1)
        self.proj = nn.Linear(input_dim, output_dim)

    def forward(self, h, mask=None):
        # h: (B, T, H)
        # mask: (B, T) binary mask (1=keep, 0=ignore)
        scores = self.attn(h).squeeze(-1)  # (B, T)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn_weights = torch.softmax(scores, dim=-1)  # (B, T)
        pooled = torch.bmm(attn_weights.unsqueeze(1), h).squeeze(1)  # (B, H)
        return self.proj(pooled), attn_weights


class DayCentModel(nn.Module):
    def __init__(self, input_dim, init_dim, year_dim, hidden_dim=128, latent_dim=32, lstm_layers=2):
        super().__init__()

        # project init conditions + year encoding → latent feature
        self.init_proj = nn.Linear(init_dim + year_dim, latent_dim)

        # project daily inputs → latent feature
        self.daily_proj = nn.Linear(input_dim, latent_dim)

        # combine both
        self.lstm = nn.LSTM(
            input_size=latent_dim * 2, 
            hidden_size=hidden_dim, 
            num_layers=lstm_layers, 
            batch_first=True
        )

        # Attention heads
        self.somsc_attn = AttentionPooling(hidden_dim, hidden_dim)
        self.yield_attn = AttentionPooling(hidden_dim, hidden_dim)

        # Output layers
        self.somsc_head = nn.Linear(hidden_dim, 1)  # monthly but gets stacked in forward pass
        self.yield_head = nn.Linear(hidden_dim, 1)   # yearly

    def forward(self, batch):
        seq = batch["sequence"]             # (B, 365, F)
        init_cond = batch["init_cond"]      # (B, I)
        year_enc = batch["year_enc"]        # (B, Y)
        harvest_mask = batch["harvest_mask"]# (B, 365)

        # ---- daily representation ----
        daily_latent = self.daily_proj(seq)  # (B, 365, latent)
        
        # ---- global representation (init+year) ----
        global_latent = self.init_proj(torch.cat([init_cond, year_enc], dim=-1))  # (B, latent)

        global_latent = global_latent.unsqueeze(1).repeat(1, seq.size(1), 1)      # (B, 365, latent)

        # ---- concat + LSTM ----
        x = torch.cat([daily_latent, global_latent], dim=-1)  # (B, 365, 2*latent)
        h, _ = self.lstm(x)                                   # (B, 365, H)

        # ---- SOMSC head ----
        somsc_preds = []
        somsc_attns = []
        ranges = month_day_ranges()
        for m, (start, end) in enumerate(ranges):
            mask = torch.zeros(h.shape[:2], dtype=torch.bool, device=h.device)
            mask[:, 0:end] = True
            pooled, attn = self.somsc_attn(h, mask=mask)
            pred = self.somsc_head(pooled)
            somsc_preds.append(pred)
            somsc_attns.append(attn)
        somsc_preds = torch.stack(somsc_preds, dim=1)  # (B,12)

        # ---- Yield head ----
        yield_repr, yield_attn = self.yield_attn(h, mask=harvest_mask)  # (B, H)
        yield_pred = self.yield_head(yield_repr).squeeze(-1)            # (B,)

        return {
            "somsc_pred": somsc_preds,
            "yield_pred": yield_pred,
            "somsc_attn": somsc_attns,
            "yield_attn": yield_attn
        }

In [19]:
# ----------------------
# Config
# ----------------------
BATCH_SIZE = 512
EPOCHS = 50
LR = 4e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="kgml",
    # Set the wandb project where this run will be logged.
    project="daycent",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": LR,
        "architecture": "LSTM with Attention",
        "dataset": "Single Scenario",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE
    },
)

CommError: Error uploading run: returned error 403: {"data":{"upsertBucket":null},"errors":[{"message":"permission denied","path":["upsertBucket"],"extensions":{"code":"PERMISSION_ERROR"}}]}

In [ ]:
# ----------------------
# Dataloader
# ----------------------
# check dataset
dataset = DayCentDataset(INPUT_NPY, OUTPUT_NPY, INIT_COND, apply_scaling=True)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

0


In [74]:
# 1) Create index array
indices = np.arange(len(dataset))

# 2) First split: train+val vs test
trainval_idx, test_idx = train_test_split(
    indices, test_size=0.20, random_state=RND_SEED, shuffle=True
)

# 3) Second split: train vs val (e.g., 80/20 of trainval)
train_idx, val_idx = train_test_split(
    trainval_idx, test_size=0.20, random_state=RND_SEED, shuffle=True
)

# 4) Wrap subsets
train_ds = Subset(dataset, train_idx)
val_ds   = Subset(dataset, val_idx)
test_ds  = Subset(dataset, test_idx)

# 5) Create loaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Dataset sizes — total: {len(dataset)}, train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}")


Dataset sizes — total: 5075, train: 3248, val: 812, test: 1015


In [75]:
# infer input dim
sample = dataset[0]
seq_feat_dim = sample["sequence"].shape[1]  # #features
init_dim = sample["init_cond"].shape[0]
year_dim = sample["year_enc"].shape[0]

print(f"Input feature dim: {seq_feat_dim}, init cond dim: {init_dim}, year enc dim: {year_dim}")

Input feature dim: 19, init cond dim: 245, year enc dim: 16


In [76]:

model = DayCentModel(input_dim=seq_feat_dim, init_dim=init_dim, year_dim=year_dim)
model.to(DEVICE)

DayCentModel(
  (init_proj): Linear(in_features=261, out_features=32, bias=True)
  (daily_proj): Linear(in_features=19, out_features=32, bias=True)
  (lstm): LSTM(64, 128, num_layers=2, batch_first=True)
  (somsc_attn): AttentionPooling(
    (attn): Linear(in_features=128, out_features=1, bias=True)
    (proj): Linear(in_features=128, out_features=128, bias=True)
  )
  (yield_attn): AttentionPooling(
    (attn): Linear(in_features=128, out_features=1, bias=True)
    (proj): Linear(in_features=128, out_features=128, bias=True)
  )
  (somsc_head): Linear(in_features=128, out_features=1, bias=True)
  (yield_head): Linear(in_features=128, out_features=1, bias=True)
)

In [77]:
# ----------------------
# Optimizer & scheduler
# ----------------------
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=5)


In [78]:
batch = next(iter(loader))
batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}


In [79]:
out = model(batch)

In [80]:
def evaluate(model, loader, device):
    model.eval()
    total_somsc_loss = 0.0
    total_yield_loss = 0.0
    total_samples = 0

    with torch.no_grad():
        for batch in loader:
            # move tensors to device
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            out = model(batch)

            # SOMSC masked MSE
            pred_somsc = out["somsc_pred"].squeeze(-1)          # (B, 12)
            target_somsc = batch["somsc"]                      # (B, 12)
            mask_somsc = batch["somsc_mask"]                   # (B, 12) -- float tensor with 1/0 (or cast it)

            mask_sum = mask_somsc.sum()
            if mask_sum.item() > 0:
                somsc_loss = ((pred_somsc - target_somsc)**2 * mask_somsc).sum() / mask_sum
            else:
                somsc_loss = torch.tensor(0.0, device=device)

            # Yield masked MSE
            pred_yield = out["yield_pred"]                     # (B,)
            target_yield = batch["yield"]                      # (B,)
            mask_yield = batch["yield_mask"]                   # (B,)

            mask_y_sum = mask_yield.sum()
            if mask_y_sum.item() > 0:
                yield_loss = ((pred_yield - target_yield)**2 * mask_yield).sum() / mask_y_sum
            else:
                yield_loss = torch.tensor(0.0, device=device)

            bs = batch["sequence"].size(0)
            total_somsc_loss += somsc_loss.item() * bs
            total_yield_loss += yield_loss.item() * bs
            total_samples += bs

    if total_samples == 0:
        return float('nan'), float('nan')
    return total_somsc_loss / total_samples, total_yield_loss / total_samples

In [ ]:
# ----------------------
# Training loop
# ----------------------
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in loader:
        # move to device
        batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        optimizer.zero_grad()
        out = model(batch)

        # --- SOMSC loss ---
        somsc_target = batch["somsc"]              # (B,12)
        somsc_mask = batch["somsc_mask"]           # (B,12)

        # compute masked MSE
        somsc_loss = ((out["somsc_pred"].squeeze(-1) - somsc_target)**2 * somsc_mask).sum() / somsc_mask.sum()

        # --- Yield loss ---
        yield_target = batch["yield"]              # (B,)
        yield_mask = batch["yield_mask"]           # (B,)
        yield_loss = ((out["yield_pred"] - yield_target)**2 * yield_mask).sum() / yield_mask.sum()

        print(f"SOMSC Loss: {somsc_loss.item():.4f}, Yield Loss: {yield_loss.item():.4f}")

        # --- total loss ---
        loss = somsc_loss + yield_loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch["sequence"].size(0)

    total_loss /= len(dataset)
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss:.4f}")
    somsc_loss, yield_loss = evaluate(model, val_loader, DEVICE)
    print(f"  Val SOMSC Loss: {somsc_loss:.4f}, Yield Loss: {yield_loss:.4f}")

    # Optional scheduler step
    scheduler.step(total_loss)


SOMSC Loss: 16363.6631, Yield Loss: 1581.6591


In [85]:
evaluate(model, test_loader, DEVICE)

(11636.003134621305, 1460.867836697583)